In [0]:
# %pip install -q mlflow
# MAGIC print("✅ mlflow installed")


## 1. Setup


In [0]:
import mlflow
import json
import os
import sys
import re

# Add src/ for imports
REPO_ROOT = "/Workspace/Users/saisandeshk@iisc.ac.in/bharat-bricks-hacks/src/nyaya_dhwani/"
_src = os.path.join(REPO_ROOT, "src")
if _src not in sys.path:
    sys.path.insert(0, _src)

EXPERIMENT_NAME = "/Users/saisandeshk@iisc.ac.in/nyaya-sahayak-rag-evaluation"

mlflow.set_experiment(EXPERIMENT_NAME)
print(f"✅ MLflow experiment: {EXPERIMENT_NAME}")

## 2. Load benchmark questions

In [0]:

import json

BENCHMARK_PATH = os.path.join(REPO_ROOT, "tests", "benchmark_questions.json")
if not os.path.exists(BENCHMARK_PATH):
    # Fallback: try relative path
    BENCHMARK_PATH = "/Workspace/Users/saisandeshk@iisc.ac.in/bharat-bricks-hacks/tests/benchmark_questions.json"

with open(BENCHMARK_PATH) as f:
    benchmark = json.load(f)

triage_questions = benchmark.get("triage_questions", [])
print(f"📋 Loaded {len(triage_questions)} triage benchmark questions")

## 3. Run evaluation and log to MLflow

In [0]:
import sys
import os

# Install required dependencies
%pip install -q 'faiss-cpu>=1.8.0' 'sentence-transformers'

# Add correct path for nyaya_dhwani imports
src_path = '/Workspace/Users/saisandeshk@iisc.ac.in/bharat-bricks-hacks/src'
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# Clear any cached imports to force fresh import
if 'nyaya_dhwani' in sys.modules:
    del sys.modules['nyaya_dhwani']
if 'nyaya_dhwani.domain_classifier' in sys.modules:
    del sys.modules['nyaya_dhwani.domain_classifier']
if 'nyaya_dhwani.retrievers' in sys.modules:
    del sys.modules['nyaya_dhwani.retrievers']
if 'nyaya_dhwani.triage_engine' in sys.modules:
    del sys.modules['nyaya_dhwani.triage_engine']

from nyaya_dhwani.domain_classifier import classify_domain
from nyaya_dhwani.retrievers import get_retriever
from nyaya_dhwani.triage_engine import build_triage_context

retriever = get_retriever()

# Evaluation metrics accumulators
domain_correct = 0
domain_total = 0
section_found = 0
section_total = 0
plan_has_helpline = 0
plan_has_filing = 0
plan_has_fee = 0
plan_total = 0
mrr_sum = 0.0
retrieval_total = 0

results = []

for q in triage_questions:
    qid = q["id"]
    query = q["question"]
    expected_domain = q.get("expected_domain", "")
    expected_sections = q.get("expected_sections", [])

    # 1. Domain classification accuracy
    domains = classify_domain(query)
    detected = domains[0].domain if domains else "unknown"
    is_domain_correct = detected == expected_domain
    domain_correct += int(is_domain_correct)
    domain_total += 1

    # 2. Retrieval — check if expected sections appear in top-5
    chunks_df = retriever.search(query, k=7)
    retrieved_text = " ".join(chunks_df["text"].fillna("").tolist()) if "text" in chunks_df.columns else ""

    sections_hit = 0
    first_rank = None
    for sec in expected_sections:
        if sec.lower() in retrieved_text.lower():
            sections_hit += 1
            # Find rank of first chunk containing this section
            if first_rank is None:
                for idx, row in chunks_df.iterrows():
                    if sec.lower() in str(row.get("text", "")).lower():
                        first_rank = row.get("rank", idx)
                        break

    if expected_sections:
        section_found += sections_hit
        section_total += len(expected_sections)

    if first_rank is not None:
        mrr_sum += 1.0 / (first_rank + 1)
        retrieval_total += 1
    elif expected_sections:
        retrieval_total += 1  # counted but 0 RR

    # 3. Triage context — check action plan presence
    triage_domains, action_plan, _ = build_triage_context(query, chunks_df)
    plan_total += 1
    if action_plan:
        plan_has_helpline += int(len(action_plan.helplines) > 0)
        plan_has_filing += int(bool(action_plan.filing_body))
        plan_has_fee += int(bool(action_plan.filing_fee))

    results.append({
        "id": qid,
        "query": query[:80],
        "expected_domain": expected_domain,
        "detected_domain": detected,
        "domain_correct": is_domain_correct,
        "sections_found": sections_hit,
        "sections_expected": len(expected_sections),
        "has_action_plan": action_plan is not None,
    })

# Compute aggregate metrics
domain_accuracy = domain_correct / max(domain_total, 1)
section_recall = section_found / max(section_total, 1)
mrr = mrr_sum / max(retrieval_total, 1)
helpline_pct = plan_has_helpline / max(plan_total, 1)
filing_pct = plan_has_filing / max(plan_total, 1)
fee_pct = plan_has_fee / max(plan_total, 1)

print(f"\n{'='*50}")
print(f"📊 Evaluation Results ({domain_total} questions)")
print(f"{'='*50}")
print(f"Domain classification accuracy: {domain_accuracy:.1%}")
print(f"Section citation recall:        {section_recall:.1%}")
print(f"MRR (Mean Reciprocal Rank):     {mrr:.3f}")
print(f"Action plan — helpline present: {helpline_pct:.1%}")
print(f"Action plan — filing body:      {filing_pct:.1%}")
print(f"Action plan — fee info:         {fee_pct:.1%}")

## 4. Log to MLflow

In [0]:
PHASE = os.environ.get("NYAYA_PHASE", "phase-2")
RETRIEVAL_BACKEND = os.environ.get("NYAYA_RETRIEVAL_BACKEND", "hybrid_faiss_bm25")
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
LLM_MODEL = os.environ.get("LLM_MODEL", "databricks-llama-4-maverick")

with mlflow.start_run(run_name=f"eval-{PHASE}") as run:
    # Tags
    mlflow.set_tag("phase", PHASE)
    mlflow.set_tag("evaluator", "automated")

    # Parameters
    mlflow.log_param("embedding_model", EMBED_MODEL)
    mlflow.log_param("embedding_dim", 384)
    mlflow.log_param("llm_model", LLM_MODEL)
    mlflow.log_param("retrieval_backend", RETRIEVAL_BACKEND)
    mlflow.log_param("top_k", 7)
    mlflow.log_param("rrf_k", 60)
    mlflow.log_param("benchmark_size", domain_total)

    # Metrics
    mlflow.log_metric("domain_accuracy", domain_accuracy)
    mlflow.log_metric("section_recall", section_recall)
    mlflow.log_metric("mrr", mrr)
    mlflow.log_metric("plan_helpline_pct", helpline_pct)
    mlflow.log_metric("plan_filing_pct", filing_pct)
    mlflow.log_metric("plan_fee_pct", fee_pct)

    # Log detailed results as artifact
    results_json = json.dumps(results, indent=2, default=str)
    with open("/tmp/eval_results.json", "w") as f:
        f.write(results_json)
    mlflow.log_artifact("/tmp/eval_results.json")

    print(f"✅ MLflow run logged: {run.info.run_id}")
    print(f"   Experiment: {EXPERIMENT_NAME}")

## 5. View experiment results

Open the MLflow Experiments UI in Databricks to see:
 - Parameter comparison across phases
 - Metric trends (accuracy improvement from Phase 0 → 1 → 2)
 - Detailed results artifacts

In [0]:
# Show all runs for comparison
runs_df = mlflow.search_runs(experiment_names=[EXPERIMENT_NAME])
if not runs_df.empty:
    display(runs_df[["run_id", "start_time", "tags.phase",
                      "params.retrieval_backend", "params.llm_model",
                      "metrics.domain_accuracy", "metrics.section_recall",
                      "metrics.mrr", "metrics.plan_helpline_pct"]].sort_values("start_time", ascending=False))
else:
    print("No runs found yet. Run cell 3 first.")